# Unified SQL Generation Experiment Notebook

This notebook runs SQL-generation experiments on 16,000+ queries using:
- **Local models** via vLLM (Llama, Qwen, DeepSeek)
- **API models** (Gemini, GPT-4, Claude)

## Sections
1. Setup & Global Configuration
2. Unified Model Wrapper (LLMWorker)
3. Input Pipeline (Three Sources)
4. Execution Loop (Checkpointing, Batching)
5. Post-Processing & Evaluation Harness
6. Final Aggregation & Visualization

---
## Section 1: Setup & Global Configuration

In [ ]:
# Cell 1.1: Infrastructure Setup
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define base paths
DRIVE_BASE = '/content/drive/MyDrive/ExpResults'
REPO_PATH = f'{DRIVE_BASE}/sql-nl'

# Install dependencies
!pip install -q vllm openai anthropic google-genai sqlglot tqdm pandas matplotlib seaborn pyyaml python-dotenv

# For gated models, login to HuggingFace
!pip install -q huggingface_hub

print("✅ Dependencies installed")

In [ ]:
# Cell 1.2: Authentication
import os
from google.colab import userdata

# Load API keys from Colab Secrets
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ OpenAI API key loaded")
except:
    print("⚠️ OPENAI_API_KEY not found in Colab secrets")

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['CLAUDE_API_KEY'] = os.environ['ANTHROPIC_API_KEY']  # Alias
    print("✅ Anthropic API key loaded")
except:
    print("⚠️ ANTHROPIC_API_KEY not found in Colab secrets")

try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print("✅ Gemini API key loaded")
except:
    print("⚠️ GEMINI_API_KEY not found in Colab secrets")

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ HuggingFace login successful")
except:
    print("⚠️ HF_TOKEN not found - gated models may not work")

In [ ]:
# Cell 1.3: Directory Structure with Timestamped Run
import os
from datetime import datetime

# Create timestamped run directory
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'{DRIVE_BASE}/runs/{RUN_TIMESTAMP}'

# Create subdirectories
INPUTS_DIR = f'{RUN_DIR}/inputs'
OUTPUTS_DIR = f'{RUN_DIR}/outputs'
LOGS_DIR = f'{RUN_DIR}/logs'

for d in [INPUTS_DIR, OUTPUTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Run directory created: {RUN_DIR}")
print(f"   📁 inputs:  {INPUTS_DIR}")
print(f"   📁 outputs: {OUTPUTS_DIR}")
print(f"   📁 logs:    {LOGS_DIR}")

In [ ]:
# Cell 1.4: Add repository to path and load config
import sys
sys.path.insert(0, REPO_PATH)

import yaml

# Load experiments configuration
with open(f'{REPO_PATH}/experiments.yaml', 'r') as f:
    EXPERIMENTS_CONFIG = yaml.safe_load(f)

print(f"✅ Loaded {len(EXPERIMENTS_CONFIG['models'])} model configurations:")
for m in EXPERIMENTS_CONFIG['models']:
    print(f"   - {m['name']} ({m['adapter_type']})")

---
## Section 2: Unified Model Wrapper (LLMWorker)

In [ ]:
# Cell 2.1: LLMWorker Class
import time
import asyncio
from typing import List, Dict, Any, Optional
from abc import ABC, abstractmethod
from tqdm.auto import tqdm

# System prompt for SQL generation (consistent across all models)
SYSTEM_PROMPT = """You are a SQL expert. Given the database schema and a natural language query,
provide ONLY the SQL code. No explanations. No preamble. Return only valid SQL."""

# Database schema context
SCHEMA_CONTEXT = """
Database Schema:
- users(id INT, username VARCHAR, email VARCHAR, signup_date DATETIME, is_verified BOOLEAN, country_code VARCHAR)
- posts(id INT, user_id INT, content TEXT, posted_at DATETIME, view_count INT)
- comments(id INT, user_id INT, post_id INT, comment_text TEXT, created_at DATETIME)
- likes(user_id INT, post_id INT, liked_at DATETIME)
- follows(follower_id INT, followee_id INT, followed_at DATETIME)
"""

class LLMWorker:
    """Unified wrapper for both local (vLLM) and API-based models."""
    
    def __init__(
        self,
        adapter_type: str,
        model_identifier: str,
        rate_limit: Optional[Dict[str, Any]] = None
    ):
        self.adapter_type = adapter_type
        self.model_identifier = model_identifier
        self.rate_limit = rate_limit or {}
        self._adapter = None
        self._initialize_adapter()
    
    def _initialize_adapter(self):
        """Initialize the appropriate adapter based on type."""
        if self.adapter_type == 'vllm':
            from src.harness.adapters.vllm import VLLMAdapter
            self._adapter = VLLMAdapter(model_name=self.model_identifier)
        elif self.adapter_type == 'gemini':
            from src.harness.adapters.gemini import GeminiAdapter
            self._adapter = GeminiAdapter(model_name=self.model_identifier)
        elif self.adapter_type == 'openai':
            from src.harness.adapters.openai import OpenAIAdapter
            self._adapter = OpenAIAdapter(model_name=self.model_identifier)
        elif self.adapter_type == 'anthropic':
            from src.harness.adapters.anthropic import AnthropicAdapter
            self._adapter = AnthropicAdapter(model_name=self.model_identifier)
        else:
            raise ValueError(f"Unknown adapter type: {self.adapter_type}")
    
    def _format_prompt(self, nl_prompt: str) -> str:
        """Format prompt with schema context and system instructions."""
        return f"{SYSTEM_PROMPT}\n\n{SCHEMA_CONTEXT}\n\nQuery: {nl_prompt}\n\nSQL:"
    
    def generate_batch(self, prompts: List[str], batch_size: int = 500) -> List[str]:
        """
        Generate responses for a batch of prompts.
        
        Args:
            prompts: List of natural language prompts
            batch_size: Chunk size for vLLM (default 500)
            
        Returns:
            List of generated SQL responses
        """
        formatted_prompts = [self._format_prompt(p) for p in prompts]
        
        if self.adapter_type == 'vllm':
            # vLLM: Process in chunks
            results = []
            for i in range(0, len(formatted_prompts), batch_size):
                chunk = formatted_prompts[i:i+batch_size]
                chunk_results = self._adapter.generate(chunk)
                results.extend(chunk_results)
            return results
        else:
            # API models: Use rate limiting
            return self._generate_with_rate_limit(formatted_prompts)
    
    def _generate_with_rate_limit(self, prompts: List[str]) -> List[str]:
        """Generate with rate limiting for API models."""
        rpm = self.rate_limit.get('requests_per_minute', 60)
        max_retries = self.rate_limit.get('max_retries', 3)
        min_interval = 60.0 / rpm
        
        results = []
        last_request_time = 0
        
        for prompt in tqdm(prompts, desc=f"Generating ({self.model_identifier})"):
            # Rate limiting
            elapsed = time.time() - last_request_time
            if elapsed < min_interval:
                time.sleep(min_interval - elapsed)
            
            # Retry logic
            for attempt in range(max_retries):
                try:
                    result = self._adapter.generate([prompt])[0]
                    results.append(result)
                    last_request_time = time.time()
                    break
                except Exception as e:
                    if attempt == max_retries - 1:
                        results.append(f"ERROR: {str(e)}")
                    else:
                        time.sleep(2 ** attempt)  # Exponential backoff
        
        return results
    
    @property
    def model_name(self) -> str:
        return self._adapter.model_name()

print("✅ LLMWorker class defined")

---
## Section 3: Input Pipeline (Three Sources)

In [ ]:
# Cell 3.1: Data Loading Functions
import json
import random
from typing import List, Dict, Any

def load_baseline_queries(path: str) -> List[Dict[str, Any]]:
    """Load baseline queries (original NL prompts)."""
    with open(path, 'r') as f:
        queries = json.load(f)
    
    tasks = []
    for q in queries:
        tasks.append({
            'job_id': f"{q['id']}_baseline_original",
            'query_id': q['id'],
            'perturbation_source': 'baseline',
            'perturbation_type': 'original',
            'input_prompt': q['nl_prompt'],
            'gold_sql': q['sql'],
            'complexity': q.get('complexity', 'unknown'),
            'tables': q.get('tables', [])
        })
    return tasks

def load_systematic_perturbations(path: str) -> List[Dict[str, Any]]:
    """Load systematic perturbations."""
    with open(path, 'r') as f:
        queries = json.load(f)
    
    tasks = []
    for q in queries:
        query_id = q['id']
        gold_sql = q['sql']
        perturbations = q.get('generated_perturbations', {})
        
        # Add original
        original = perturbations.get('original', {})
        if original.get('nl_prompt'):
            tasks.append({
                'job_id': f"{query_id}_systematic_original",
                'query_id': query_id,
                'perturbation_source': 'systematic',
                'perturbation_type': 'original',
                'input_prompt': original['nl_prompt'],
                'gold_sql': gold_sql,
                'complexity': q.get('complexity', 'unknown'),
                'tables': q.get('tables', [])
            })
        
        # Add single perturbations
        for p in perturbations.get('single_perturbations', []):
            if p.get('applicable') and p.get('perturbed_nl_prompt'):
                p_type = p['perturbation_name']
                tasks.append({
                    'job_id': f"{query_id}_systematic_{p_type}",
                    'query_id': query_id,
                    'perturbation_source': 'systematic',
                    'perturbation_type': p_type,
                    'input_prompt': p['perturbed_nl_prompt'],
                    'gold_sql': gold_sql,
                    'complexity': q.get('complexity', 'unknown'),
                    'tables': q.get('tables', [])
                })
    
    return tasks

def load_llm_perturbations(path: str) -> List[Dict[str, Any]]:
    """Load LLM-generated perturbations."""
    with open(path, 'r') as f:
        queries = json.load(f)
    
    tasks = []
    for q in queries:
        query_id = q['id']
        gold_sql = q['sql']
        perturbations = q.get('generated_perturbations', {})
        
        # Add single perturbations
        for p in perturbations.get('single_perturbations', []):
            if p.get('applicable') and p.get('perturbed_nl_prompt'):
                p_type = p['perturbation_name']
                tasks.append({
                    'job_id': f"{query_id}_llm_{p_type}",
                    'query_id': query_id,
                    'perturbation_source': 'llm',
                    'perturbation_type': p_type,
                    'input_prompt': p['perturbed_nl_prompt'],
                    'gold_sql': gold_sql,
                    'complexity': q.get('complexity', 'unknown'),
                    'tables': q.get('tables', [])
                })
        
        # Add compound perturbation
        compound = perturbations.get('compound_perturbation', {})
        if compound.get('perturbed_nl_prompt'):
            tasks.append({
                'job_id': f"{query_id}_llm_compound",
                'query_id': query_id,
                'perturbation_source': 'llm',
                'perturbation_type': 'compound',
                'input_prompt': compound['perturbed_nl_prompt'],
                'gold_sql': gold_sql,
                'complexity': q.get('complexity', 'unknown'),
                'tables': q.get('tables', [])
            })
    
    return tasks

print("✅ Data loading functions defined")

In [ ]:
# Cell 3.2: Load and Merge All Data Sources
DATASET_DIR = f'{REPO_PATH}/dataset/current'

# Load all three sources
baseline_tasks = load_baseline_queries(f'{DATASET_DIR}/nl_social_media_queries_20.json')
systematic_tasks = load_systematic_perturbations(f'{DATASET_DIR}/nl_social_media_queries_systematic_20.json')
llm_tasks = load_llm_perturbations(f'{DATASET_DIR}/nl_social_media_queries_llm_perturbed_20.json')

# Merge all tasks
all_tasks = baseline_tasks + systematic_tasks + llm_tasks

# Shuffle to prevent caching bias
random.seed(42)
random.shuffle(all_tasks)

print(f"\n📊 Dataset Summary:")
print(f"   Baseline:   {len(baseline_tasks):,} tasks")
print(f"   Systematic: {len(systematic_tasks):,} tasks")
print(f"   LLM:        {len(llm_tasks):,} tasks")
print(f"   ────────────────────")
print(f"   Total:      {len(all_tasks):,} tasks")

# Save flat tasks to inputs directory
flat_tasks_path = f'{INPUTS_DIR}/flat_tasks.jsonl'
with open(flat_tasks_path, 'w') as f:
    for task in all_tasks:
        f.write(json.dumps(task) + '\n')

print(f"\n✅ Saved to: {flat_tasks_path}")

---
## Section 4: Execution Loop

In [ ]:
# Cell 4.1: Experiment Runner Class
import json
import os
from typing import Set, Tuple
from tqdm.auto import tqdm
from datetime import datetime

class ExperimentRunner:
    """Manages experiment execution with checkpointing and progress tracking."""
    
    def __init__(self, output_path: str):
        self.output_path = output_path
        self.processed_keys: Set[Tuple[str, str]] = set()
        self._load_checkpoint()
    
    def _load_checkpoint(self):
        """Load already-processed (job_id, model_name) pairs."""
        if os.path.exists(self.output_path):
            with open(self.output_path, 'r') as f:
                for line in f:
                    if line.strip():
                        try:
                            record = json.loads(line)
                            key = (record['job_id'], record['model_name'])
                            self.processed_keys.add(key)
                        except json.JSONDecodeError:
                            pass
            print(f"📋 Loaded checkpoint: {len(self.processed_keys):,} processed pairs")
    
    def get_pending_tasks(self, tasks: List[Dict], model_name: str) -> List[Dict]:
        """Filter tasks to only those not yet processed for this model."""
        pending = []
        for task in tasks:
            key = (task['job_id'], model_name)
            if key not in self.processed_keys:
                pending.append(task)
        return pending
    
    def run(
        self,
        tasks: List[Dict],
        worker: LLMWorker,
        batch_size: int = 500,
        limit: int = None
    ):
        """Run experiment for a specific model."""
        model_name = worker.model_name
        pending = self.get_pending_tasks(tasks, model_name)
        
        if limit:
            pending = pending[:limit]
        
        if not pending:
            print(f"✅ All tasks already processed for {model_name}")
            return
        
        print(f"\n🚀 Running {len(pending):,} tasks for {model_name}")
        
        # Extract prompts
        prompts = [t['input_prompt'] for t in pending]
        
        # Generate responses
        start_time = datetime.now()
        responses = worker.generate_batch(prompts, batch_size=batch_size)
        elapsed = (datetime.now() - start_time).total_seconds()
        
        print(f"⏱️ Generation completed in {elapsed:.1f}s ({len(responses)/elapsed:.1f} queries/sec)")
        
        # Save results
        with open(self.output_path, 'a') as f:
            for task, response in zip(pending, responses):
                result = {
                    **task,
                    'model_name': model_name,
                    'generated_response': response,
                    'timestamp': datetime.now().isoformat()
                }
                f.write(json.dumps(result) + '\n')
                self.processed_keys.add((task['job_id'], model_name))
        
        print(f"💾 Results saved to {self.output_path}")

# Initialize runner
RESULTS_PATH = f'{OUTPUTS_DIR}/results.jsonl'
runner = ExperimentRunner(RESULTS_PATH)
print("\n✅ ExperimentRunner initialized")

In [ ]:
# Cell 4.2: Run Experiments for Selected Model
# ⚠️ CONFIGURE THIS CELL before running!

# Select which model to run (uncomment ONE):
# MODEL_NAME = 'gemini-2.5-flash-lite'
# MODEL_NAME = 'gpt-4o'
# MODEL_NAME = 'claude-4.5'
# MODEL_NAME = 'local-qwen0.5b'
# MODEL_NAME = 'llama3.1-8b'
# MODEL_NAME = 'deepseek-coder-v2-lite'

MODEL_NAME = 'gemini-2.5-flash-lite'  # Default

# Find model config
model_config = None
for m in EXPERIMENTS_CONFIG['models']:
    if m['name'] == MODEL_NAME:
        model_config = m
        break

if model_config is None:
    raise ValueError(f"Model {MODEL_NAME} not found in experiments.yaml")

print(f"📌 Selected model: {MODEL_NAME}")
print(f"   Adapter: {model_config['adapter_type']}")
print(f"   Identifier: {model_config['model_identifier']}")

# Initialize worker
worker = LLMWorker(
    adapter_type=model_config['adapter_type'],
    model_identifier=model_config['model_identifier'],
    rate_limit=model_config.get('rate_limit')
)

# Run experiment (set limit for testing)
runner.run(all_tasks, worker, limit=None)  # Set limit=10 for testing

---
## Section 5: Post-Processing & Evaluation Harness

In [ ]:
# Cell 5.1: SQL Extraction Function
import re

def extract_sql(text: str) -> str:
    """
    Extract SQL from LLM response.
    
    Priority:
    1. Extract from ```sql...``` blocks
    2. Extract from ```...``` blocks
    3. Find first SELECT/INSERT/UPDATE/DELETE to last ;
    4. Return cleaned raw text
    """
    if not text or text.startswith('ERROR:'):
        return ''
    
    # Try ```sql...``` block
    sql_block = re.search(r'```sql\s*(.+?)```', text, re.DOTALL | re.IGNORECASE)
    if sql_block:
        return sql_block.group(1).strip()
    
    # Try generic code block
    code_block = re.search(r'```\s*(.+?)```', text, re.DOTALL)
    if code_block:
        return code_block.group(1).strip()
    
    # Find SQL keywords
    sql_pattern = r'(SELECT|INSERT|UPDATE|DELETE|WITH)\s+.+?;'
    match = re.search(sql_pattern, text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    
    # Last resort: clean and return
    return text.strip()

# Test extraction
test_cases = [
    "```sql\nSELECT * FROM users;\n```",
    "Here is the SQL:\nSELECT id FROM posts WHERE view_count > 100;",
    "SELECT name FROM users"
]
print("Testing SQL extraction:")
for tc in test_cases:
    print(f"  Input: {tc[:50]}...")
    print(f"  Output: {extract_sql(tc)}\n")

In [ ]:
# Cell 5.2: Equivalence Testing
from src.equivalence.equivalence_engine import SQLEquivalenceEngine
from src.equivalence.config import EquivalenceConfig

# Initialize equivalence engine
eq_config = EquivalenceConfig(
    base_db_path=f'{REPO_PATH}/test_dbs/base.sqlite',
    test_suite_dir=f'{REPO_PATH}/test_dbs'
)
eq_engine = SQLEquivalenceEngine(eq_config)

print("✅ Equivalence engine initialized")

In [ ]:
# Cell 5.3: Run Evaluation on Results
import json
from tqdm.auto import tqdm

EVAL_RESULTS_PATH = f'{OUTPUTS_DIR}/evaluated_results.jsonl'

def evaluate_results(input_path: str, output_path: str):
    """Run equivalence testing on all results."""
    results = []
    
    # Load results
    with open(input_path, 'r') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
    
    print(f"Evaluating {len(results):,} results...")
    
    evaluated = []
    for r in tqdm(results):
        # Extract SQL
        generated_sql = extract_sql(r.get('generated_response', ''))
        gold_sql = r['gold_sql']
        
        # Check equivalence
        try:
            eq_result = eq_engine.check_equivalence(gold_sql, generated_sql)
            is_equivalent = eq_result.is_equivalent
            eq_details = eq_result.details
        except Exception as e:
            is_equivalent = False
            eq_details = f"Error: {str(e)}"
        
        # Add evaluation fields
        r['generated_sql'] = generated_sql
        r['is_equivalent'] = is_equivalent
        r['equivalence_details'] = eq_details
        evaluated.append(r)
    
    # Save evaluated results
    with open(output_path, 'w') as f:
        for r in evaluated:
            f.write(json.dumps(r) + '\n')
    
    # Summary
    total = len(evaluated)
    passed = sum(1 for r in evaluated if r['is_equivalent'])
    print(f"\n📊 Evaluation Summary:")
    print(f"   Total: {total:,}")
    print(f"   Passed: {passed:,} ({100*passed/total:.1f}%)")
    print(f"   Failed: {total-passed:,} ({100*(total-passed)/total:.1f}%)")
    print(f"\n💾 Saved to: {output_path}")
    
    return evaluated

# Run evaluation
if os.path.exists(RESULTS_PATH):
    evaluated_results = evaluate_results(RESULTS_PATH, EVAL_RESULTS_PATH)
else:
    print("⚠️ No results found. Run Section 4 first.")

---
## Section 6: Final Aggregation & Visualization

In [ ]:
# Cell 6.1: Load Evaluated Results into DataFrame
import pandas as pd
import json

def load_results_df(path: str) -> pd.DataFrame:
    """Load evaluated results into a DataFrame."""
    records = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return pd.DataFrame(records)

if os.path.exists(EVAL_RESULTS_PATH):
    df = load_results_df(EVAL_RESULTS_PATH)
    print(f"Loaded {len(df):,} evaluated results")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nModels: {df['model_name'].unique().tolist()}")
    print(f"Sources: {df['perturbation_source'].unique().tolist()}")
else:
    print("⚠️ No evaluated results found. Run Section 5 first.")

In [ ]:
# Cell 6.2: Visualization Functions
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

def plot_accuracy_by_model(df: pd.DataFrame):
    """Bar chart: Accuracy by model."""
    acc = df.groupby('model_name')['is_equivalent'].mean() * 100
    acc = acc.sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(acc.index, acc.values, color=sns.color_palette('viridis', len(acc)))
    ax.set_xlabel('Accuracy (%)')
    ax.set_title('SQL Generation Accuracy by Model')
    ax.set_xlim(0, 100)
    
    for bar, val in zip(bars, acc.values):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/accuracy_by_model.png', dpi=150)
    plt.show()

def plot_accuracy_by_source(df: pd.DataFrame):
    """Bar chart: Accuracy by perturbation source."""
    acc = df.groupby('perturbation_source')['is_equivalent'].mean() * 100
    
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {'baseline': '#2ecc71', 'systematic': '#3498db', 'llm': '#9b59b6'}
    bars = ax.bar(acc.index, acc.values, color=[colors.get(s, '#95a5a6') for s in acc.index])
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Perturbation Source')
    ax.set_ylim(0, 100)
    
    for bar, val in zip(bars, acc.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 2, f'{val:.1f}%', ha='center')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/accuracy_by_source.png', dpi=150)
    plt.show()

def plot_accuracy_heatmap(df: pd.DataFrame):
    """Heatmap: Accuracy by model and perturbation type."""
    pivot = df.pivot_table(
        values='is_equivalent',
        index='perturbation_type',
        columns='model_name',
        aggfunc='mean'
    ) * 100
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100,
                cbar_kws={'label': 'Accuracy (%)'})
    ax.set_title('Accuracy by Model and Perturbation Type')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/accuracy_heatmap.png', dpi=150)
    plt.show()

def plot_accuracy_by_complexity(df: pd.DataFrame):
    """Grouped bar chart: Accuracy by query complexity."""
    pivot = df.pivot_table(
        values='is_equivalent',
        index='complexity',
        columns='model_name',
        aggfunc='mean'
    ) * 100
    
    pivot.plot(kind='bar', figsize=(12, 6), colormap='viridis')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy by Query Complexity')
    plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/accuracy_by_complexity.png', dpi=150)
    plt.show()

def plot_error_distribution(df: pd.DataFrame):
    """Pie chart: Distribution of error types."""
    failed = df[~df['is_equivalent']].copy()
    
    # Categorize errors
    def categorize_error(details):
        if not details:
            return 'Unknown'
        details = str(details).lower()
        if 'parse' in details:
            return 'Parse Error'
        elif 'type mismatch' in details:
            return 'Type Mismatch'
        elif 'not equivalent' in details:
            return 'Semantic Mismatch'
        else:
            return 'Other'
    
    failed['error_type'] = failed['equivalence_details'].apply(categorize_error)
    error_counts = failed['error_type'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    colors = ['#e74c3c', '#f39c12', '#3498db', '#95a5a6']
    ax.pie(error_counts.values, labels=error_counts.index, autopct='%1.1f%%',
           colors=colors[:len(error_counts)])
    ax.set_title('Distribution of Error Types')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/error_distribution.png', dpi=150)
    plt.show()

def plot_hardest_queries(df: pd.DataFrame, top_n: int = 20):
    """Bar chart: Queries with lowest success rate."""
    query_acc = df.groupby('query_id')['is_equivalent'].mean() * 100
    hardest = query_acc.nsmallest(top_n)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh([f'Q{q}' for q in hardest.index], hardest.values, color='#e74c3c')
    ax.set_xlabel('Success Rate (%)')
    ax.set_title(f'Top {top_n} Hardest Queries')
    ax.set_xlim(0, 100)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/hardest_queries.png', dpi=150)
    plt.show()

print("✅ Visualization functions defined")

In [ ]:
# Cell 6.3: Generate All Plots
if 'df' in dir() and len(df) > 0:
    print("📊 Generating visualizations...\n")
    
    plot_accuracy_by_model(df)
    plot_accuracy_by_source(df)
    plot_accuracy_heatmap(df)
    plot_accuracy_by_complexity(df)
    plot_error_distribution(df)
    plot_hardest_queries(df)
    
    print(f"\n✅ All plots saved to {OUTPUTS_DIR}")
else:
    print("⚠️ No data to visualize. Run previous sections first.")

In [ ]:
# Cell 6.4: Summary Report
if 'df' in dir() and len(df) > 0:
    print("="*60)
    print("EXPERIMENT SUMMARY REPORT")
    print("="*60)
    print(f"\nRun: {RUN_TIMESTAMP}")
    print(f"Total evaluations: {len(df):,}")
    print(f"\nOverall accuracy: {df['is_equivalent'].mean()*100:.2f}%")
    
    print("\n" + "-"*40)
    print("BY MODEL:")
    print("-"*40)
    model_acc = df.groupby('model_name')['is_equivalent'].agg(['sum', 'count', 'mean'])
    model_acc.columns = ['Passed', 'Total', 'Accuracy']
    model_acc['Accuracy'] = (model_acc['Accuracy'] * 100).round(2)
    print(model_acc.to_string())
    
    print("\n" + "-"*40)
    print("BY PERTURBATION SOURCE:")
    print("-"*40)
    source_acc = df.groupby('perturbation_source')['is_equivalent'].agg(['sum', 'count', 'mean'])
    source_acc.columns = ['Passed', 'Total', 'Accuracy']
    source_acc['Accuracy'] = (source_acc['Accuracy'] * 100).round(2)
    print(source_acc.to_string())
    
    print("\n" + "="*60)